In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import numpy as np
import pandas as pd
import re
from pathlib import Path
from lifelines import CoxPHFitter
from gbmcollaboration import AllDataCenters
from gbmcollaboration.viz import set_mpl_rcparams, savefig
set_mpl_rcparams()

# Paths

In [ ]:
CELL_TYPE_MAPPING_CSV = '../../gbmcollaboration/metadata/SingleCell_Annotation_Key_v3b.csv'
CELL_TYPE_INFO_XLSX = '../../gbmcollaboration/metadata/Cell type annotation markers Glimmer.xlsx'
FIGURES_DIR = Path('/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/figures/scRNAseq/annotation/')

# Load data

In [ ]:
# load sc data
ad = AllDataCenters.load_singlecell(
    exclude_idh1_mut = True,
    exclude_patient_ids = True,
    exclude_relapse = True
)

# ensure raw counts are in .X. clear log1p status if present
if 'log1p' in ad.uns:
    del ad.uns['log1p']
ad.X = ad.layers['unnormalized_counts']
sc.pp.normalize_total(ad)
sc.pp.log1p(ad)

# load cell type name mapping
# df_ct_map = pd.read_csv(CELL_TYPE_MAPPING_CSV)
df_markers, df_ct_map = pd.read_excel(CELL_TYPE_INFO_XLSX,sheet_name=None).values()
df_markers['Level 4 Markers (WIP)'] = df_markers['Level 4 Markers (WIP)'].astype(str)
markers_map = df_markers[['celltype_level4_curated','Level 4 Markers (WIP)']].dropna().set_index('celltype_level4_curated').squeeze()
markers_map = markers_map.apply(lambda x: [s.strip() for s in x.split(',')]).to_dict()

# create columns with simplified cell type names
for i in range(1,5):
    col = f'celltype_level{i}_curated'
    _map = df_ct_map.set_index('celltype_level4_scanvi')[col]
    ad.obs[col] = ad.obs['celltype_level4_scanvi'].astype(str).replace(_map)

ct_groups_lvl1_custom = {
    'Malignant': 'Malignant',
    'Other': 'Other',
    'Neuron': 'Stromal',
    'Glia': 'Stromal',
    'Vascular': 'Stromal',
    'Lymphoid': 'Immune',
    'Myeloid': 'Immune'
}
ad.obs['celltype_level1_custom'] = ad.obs['celltype_level1_curated'].replace(ct_groups_lvl1_custom)

# exclude cell types
exclude_celltypes = [
    'Perivascular_fibroblast_cycling',
    'cDC2_cycling'
]

ad = ad[~ad.obs['celltype_level4_curated'].isin(exclude_celltypes)].copy()

In [ ]:
exclude_celltypes = [
    'Perivascular_fibroblast_cycling',
    'cDC2_cycling'
]

ad = ad[~ad.obs['celltype_level4_curated'].isin(exclude_celltypes)]

In [ ]:
celltype_levels = [
    'celltype_level1_custom',
    'celltype_level1_curated',
    # 'celltype_level2_curated',
    # 'celltype_level3_curated',
]

for level in celltype_levels:

    groups = np.unique(ad.obs[level])
    
    for group_name in groups:
        print(f"Processing {level} {group_name}")

        if any(s in group_name for s in ['Doublet','Other']):
            continue

        # subset AnnData to just this group
        ad_sub = ad[ad.obs[level] == group_name].copy()
        ad_sub.obs['celltype_level4_curated'] = ad_sub.obs['celltype_level4_curated'].astype(str)

        u_ct = np.unique(ad_sub.obs['celltype_level4_curated'])

        markers_map_grp = {}
        for ct in u_ct:
            ct_gene_list = markers_map[ct]
            markers_map_grp[ct] = []
            for g in ct_gene_list:
                g_clean = re.sub(r'\s*\(.*\)', '', g).strip()
                if g_clean in ad.var_names:
                    markers_map_grp[ct].append(g_clean)
                else:
                    print("\t",ct,"missing gene:", g_clean)


        genes_to_use = [g for genes in markers_map_grp.values() for g in genes]
        unique_genes_to_use = list(dict.fromkeys(genes_to_use))

        mask = np.isin(ad_sub.var_names, unique_genes_to_use)

        if not len(unique_genes_to_use):
            print("Skipping", group_name)
            continue

        # rank genes for this group
        # sc.tl.rank_genes_groups(ad_sub, groupby=level, mask_var=mask)

        # create dotplot (or any plot)
        # fig = sc.pl.rank_genes_groups_dotplot(ad_sub, standard_scale='var', var_names=markers_map_grp,return_fig=True)

        fig = sc.pl.dotplot(
            ad_sub,
            var_names=unique_genes_to_use,
            groupby='celltype_level4_curated', 
            categories_order=u_ct,   
            standard_scale='var', 
            return_fig=True
        )
        # fig.show()
        
        # save plot
        out_path = Path(FIGURES_DIR) / f'dotplot_{level}_{group_name}.png'
        out_path.parent.mkdir(parents=True, exist_ok=True)
        savefig(None, out_path, fig=fig, close=False)
        
        # delete subset to free memory
        del ad_sub